# Laterality score for HESTA and MOSTA sagittal sections

**Goal.** Estimate how far each sagittal section lies from the body midline,
so that HESTA (human) and MOSTA (mouse) sections at matched developmental
stages can be paired by comparable mediolateral position.

Neither dataset stores a section coordinate along the left–right axis, so the
position is inferred from *what the section cuts through*:

1. **Composition depth `D`** (0 = midline, 1 = lateral). Every annotated organ
   label is mapped to an anatomical depth class:
   - *axial* (depth 0): structures that only exist on the midline – spinal cord,
     notochord, cloaca, pituitary, …
   - *paramedian* (depth 0.5): paired structures next to the midline – kidneys
     (meso/meta-nephros), gonads, adrenals, lungs, dorsal root ganglia,
     myotome/dermomyotome, …
   - *lateral* (depth 1): structures far from the midline – limb buds, eye,
     inner ear, …
   - *ignored*: labels present at every depth (cavity, blood, brain, liver,
     skin, mesenchyme, …).

   `D` is the area-weighted mean depth over all non-ignored spots.
2. **Axial coverage `C`** (0 – 1). Geometric check: the length of the largest
   connected piece of axial tissue (spinal cord / notochord), measured along the
   tissue (geodesic length), divided by the geodesic length of the whole
   section. A midline section cuts the spinal cord along its full length
   (`C` large); a section a few hundred µm off-midline misses it or only grazes
   it (`C` small). In curled embryos the path through the tissue can short-cut
   from head to tail, so the ratio is capped at 1 (`C_raw` keeps the uncapped
   value).
3. **Combined laterality `L = w_D·D + (1−w_D)·(1−C)`** in [0, 1].

**Left vs. right.** A section at +d and −d from the midline has almost the same
composition, so `L` cannot tell the sides apart (the mirror symmetry from
Problem 2). As an exploratory *side hint* we report the fraction of heart and
gut spots expressing **PITX2/Pitx2**, the classic left-side determinant
(left atrium, left sinus horn, left gut/stomach). High values suggest a left
section, values near zero a right section; near-midline sections show both.

**Caveats.**
- Scores are comparable *within an embryo* (ordering of sections) more reliably
  than *across stages*, because body proportions change with development.
- Label vocabularies differ between datasets and stages. The notebook prints
  every label that is not in the depth dictionary – extend `DEPTH_CLASS` when
  new files are added.
- Only the label column, `obsm['spatial']`, the gene names and (optionally) one
  gene column are read, directly with `h5py` (chunked scan for the gene), so the
  multi-GB files are never loaded into memory. anndata's backed mode is avoided
  because it still loads every layer (e.g. `counts`) into memory.

In [1]:
import sys
sys.path.append("..")

import re
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage, sparse
from scipy.sparse import csgraph
from scipy.spatial import cKDTree

import anndata as ad
from mgw.config import get_data_dir

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

## Configuration

In [2]:
# Folders holding the .h5ad files. None: the hesta/ and mouse_embryo/ subfolders of the
# MGW data directory (from get_data_dir(), as in demos/hesta_align.ipynb).
HESTA_DIR = None   # e.g. "/path/to/hesta_downloads"
MOSTA_DIR = None   # e.g. "/path/to/mosta_downloads"

if HESTA_DIR is None:
    HESTA_DIR = get_data_dir() / "hesta"  # asks on first run, then remembered per machine
if MOSTA_DIR is None:
    MOSTA_DIR = get_data_dir() / "mouse_embryo"
HESTA_DIR = Path(HESTA_DIR).expanduser()
MOSTA_DIR = Path(MOSTA_DIR).expanduser()
print("HESTA data directory:", HESTA_DIR)
print("MOSTA data directory:", MOSTA_DIR)

OUT_DIR = Path("laterality")  # results are written next to this notebook
OUT_DIR.mkdir(exist_ok=True)

DATASETS = {
    # name: (folder, glob pattern, obs column with organ labels, filename regex)
    "HESTA": (HESTA_DIR, "*_HESTA.h5ad", "celltype",
              r"^(?P<stage>CS[\d\-–]+)_E(?P<embryo>\d+)S(?P<section>\d+)"),
    "MOSTA": (MOSTA_DIR, "*.MOSTA.h5ad", "annotation",
              r"^(?P<stage>E[\d.]+)_E(?P<embryo>\d+)S(?P<section>\d+)"),
}
for ds, (folder, pattern, *_) in DATASETS.items():
    files = sorted(folder.glob(pattern))
    if not files:
        raise FileNotFoundError(f"no {pattern} files in {folder}. Set {ds}_DIR above, or call "
                                "get_data_dir(reset=True) to change the data directory.")
    print(f"{ds}: {len(files)} sections: {', '.join(f.name for f in files)}")

# Gene used for the left/right side hint, and the organs in which it is evaluated.
SIDE_GENE = {"HESTA": "PITX2", "MOSTA": "Pitx2"}
SIDE_ORGANS = ["heart", "gut", "gi tract", "stomach"]
COMPUTE_SIDE_HINT = True  # scans X once per file (chunked); set False to skip

W_COMPOSITION = 0.5  # weight w_D of the composition depth in the combined score

# Stage correspondence (HESTA sample -> candidate MOSTA days), from the stage table.
STAGE_PAIRS = {
    "CS12-13": ["E9.5"],
    "CS14-15": ["E10.5"],
    "CS17": ["E11.5", "E12.5"],
    "CS18": ["E12.5"],
    "CS19": ["E12.5", "E13.5"],
    "CS20": ["E13.5"],
    "CS23": ["E14.5"],
}

HESTA data directory: /nas/lnt/ga79jin/MGW/data/hesta
MOSTA data directory: /nas/lnt/ga79jin/MGW/data/mouse_embryo
HESTA: 27 sections: CS12-13_E2S1_HESTA.h5ad, CS12-13_E2S3_HESTA.h5ad, CS12-13_E2S4_HESTA.h5ad, CS14-15_E2S10_HESTA.h5ad, CS14-15_E2S6_HESTA.h5ad, CS14-15_E2S7_HESTA.h5ad, CS14-15_E2S8_HESTA.h5ad, CS14-15_E2S9_HESTA.h5ad, CS17_E1S2_HESTA.h5ad, CS17_E1S7_HESTA.h5ad, CS17_E1S8_HESTA.h5ad, CS17_E1S9_HESTA.h5ad, CS18_E1S1_HESTA.h5ad, CS18_E1S2_HESTA.h5ad, CS18_E1S3_HESTA.h5ad, CS18_E1S4_HESTA.h5ad, CS19_E1S2_HESTA.h5ad, CS19_E2S2_HESTA.h5ad, CS19_E2S3_HESTA.h5ad, CS19_E2S4_HESTA.h5ad, CS20_E1S5_HESTA.h5ad, CS20_E1S6_HESTA.h5ad, CS20_E1S7_HESTA.h5ad, CS23_E2S5_HESTA.h5ad, CS23_E2S6_HESTA.h5ad, CS23_E2S7_HESTA.h5ad, CS23_E2S8_HESTA.h5ad
MOSTA: 29 sections: E10.5_E1S1.MOSTA.h5ad, E10.5_E1S2.MOSTA.h5ad, E10.5_E2S1.MOSTA.h5ad, E11.5_E1S1.MOSTA.h5ad, E11.5_E1S2.MOSTA.h5ad, E11.5_E1S3.MOSTA.h5ad, E11.5_E1S4.MOSTA.h5ad, E12.5_E1S1.MOSTA.h5ad, E12.5_E1S2.MOSTA.h5ad, E13.5_E1S1.MOSTA.h5a

### Organ label -> depth class

Keys are lower-case label names (both datasets). Classes: `axial`, `paramedian`,
`lateral`, `ignore`. Unknown labels are reported below and treated as `ignore`.

In [3]:
DEPTH_VALUE = {"axial": 0.0, "paramedian": 0.5, "lateral": 1.0}

DEPTH_CLASS = {
    # --- axial: exist only on / at the midline
    "spinal cord": "axial",
    "notochord": "axial",
    "cloaca": "axial",
    "pituitary": "axial",
    "pituitary gland": "axial",
    "tongue": "axial",
    "thyroid": "axial",
    "esophagus": "axial",
    "trachea": "axial",
    "bladder": "axial",
    # --- paramedian: paired structures beside the midline
    "mesonephron": "paramedian",
    "mesonephros": "paramedian",
    "pronephron": "paramedian",
    "kidney": "paramedian",
    "metanephros": "paramedian",
    "urogenital ridge": "paramedian",
    "gonad": "paramedian",
    "testis": "paramedian",
    "ovary": "paramedian",
    "adrenal gland": "paramedian",
    "lung": "paramedian",
    "lung primordium": "paramedian",
    "ganglion": "paramedian",
    "dorsal root ganglion": "paramedian",
    "sympathetic nerve": "paramedian",
    "myotome": "paramedian",
    "dermomyotome": "paramedian",
    "thymus": "paramedian",
    "olfactory epithelium": "paramedian",
    # --- lateral: far from the midline
    "limb bud": "lateral",
    "limb": "lateral",
    "eye": "lateral",
    "lens": "lateral",
    "retina": "lateral",
    "inner ear": "lateral",
    "otic vesicle": "lateral",
    "submandibular gland": "lateral",
    "spleen": "lateral",
    # --- ignore: present at every depth / not informative
    "cavity": "ignore",
    "extraembryonic tissue": "ignore",
    "erythroblasts": "ignore",
    "blood vessel": "ignore",
    "agm": "ignore",
    "brain": "ignore",
    "meninges": "ignore",
    "choroid plexus": "ignore",
    "heart": "ignore",
    "liver": "ignore",
    "gut": "ignore",
    "gi tract": "ignore",
    "stomach": "ignore",
    "pancreas": "ignore",
    "mesentery": "ignore",
    "epidermis": "ignore",
    "surface ectoderm": "ignore",
    "connective tissue": "ignore",
    "mesenchyme": "ignore",
    "head mesenchyme": "ignore",
    "neural crest": "ignore",
    "sclerotome": "ignore",
    "cartilage": "ignore",
    "cartilage primordium": "ignore",
    "bone": "ignore",
    "cranial skeleton": "ignore",
    "skeletal muscle": "ignore",
    "muscle": "ignore",
    "smooth muscle": "ignore",
    "adipose tissue": "ignore",
    "branchial arch": "ignore",
    "jaw and tooth": "ignore",
    "mucosal epithelium": "ignore",
    "facial nerve": "ignore",
}

# Labels that are not tissue of the embryo (excluded from the tissue mask).
NON_TISSUE = {"cavity", "extraembryonic tissue"}
AXIAL_LABELS = {k for k, v in DEPTH_CLASS.items() if v == "axial"}

## Loading (obs labels + coordinates only)

In [4]:
def _strings(ds):
    return np.asarray(ds.asstr()[:], dtype=str)


def read_section(path, label_col):
    """Return (DataFrame with x, y, label; gene names) for one .h5ad file.

    Reads only these three elements with h5py. anndata's backed mode would also
    load every layer (e.g. counts) into memory, and AnnData objects are freed only
    by the cyclic garbage collector, so several of them can pile up across files.
    Handles the current categorical encoding and the older on-disk layout of the
    MOSTA files (codes referencing obs/__categories).
    """
    with h5py.File(path, "r") as h:
        col = h["obs"][label_col]
        if isinstance(col, h5py.Group):  # categorical: codes + categories
            codes, cats = col["codes"][:], _strings(col["categories"])
        elif "categories" in col.attrs:  # old layout
            codes, cats = col[:], _strings(h[col.attrs["categories"]])
        else:  # plain strings
            codes, cats = None, None
        labels = _strings(col) if codes is None else np.where(codes >= 0, cats[codes], "nan")
        xy = np.asarray(h["obsm"]["spatial"][:, :2], dtype=float)
        genes = _strings(h["var"][h["var"].attrs["_index"]])
    return pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "label": labels}), genes


def gene_vector(path, gene, genes, layer=None):
    """Expression of one gene for all spots, read from a CSR matrix in chunks."""
    hit = np.flatnonzero(genes == gene)
    if hit.size == 0:
        return None
    col = hit[0]
    with h5py.File(path, "r") as h:
        M = h["X"] if layer is None else h["layers"][layer]
        fmt = M.attrs.get("encoding-type", M.attrs.get("h5sparse_format", ""))
        if fmt not in ("csr_matrix", "csr"):
            raise ValueError(f"expected a CSR matrix, got {fmt!r}")
        indptr = M["indptr"][:]
        n_obs = len(indptr) - 1
        out = np.zeros(n_obs, dtype=np.float32)
        step = 2000
        for r0 in range(0, n_obs, step):
            r1 = min(r0 + step, n_obs)
            a, b = indptr[r0], indptr[r1]
            idx = M["indices"][a:b]
            sel = np.flatnonzero(idx == col)
            if sel.size:
                dat = M["data"][a:b][sel]
                rows = np.searchsorted(indptr[r0:r1 + 1], a + sel, side="right") - 1 + r0
                out[rows] = dat
    return out


def parse_name(path, regex):
    m = re.match(regex, Path(path).name)
    if m is None:
        return {"stage": "?", "embryo": "?", "section": "?"}
    d = m.groupdict()
    d["stage"] = d["stage"].replace("–", "-")
    return d

## Features

In [5]:
MAX_RASTER_PIXELS = 50_000_000  # guard against a wrongly estimated grid step


def rasterize(df):
    """Map spot coordinates to integer grid indices (one pixel per bin).

    Bin spacing and grid orientation are estimated from nearest-neighbour vectors,
    so sections whose grid is rotated (e.g. MOSTA E11.5_E1S2, rotated by 10°) are
    turned back onto the axes instead of producing a huge raster.
    """
    xy = df[["x", "y"]].to_numpy(float)
    dist, nn = cKDTree(xy).query(xy, k=2)
    step = np.median(dist[:, 1])
    v = xy[nn[:, 1]] - xy
    theta = np.angle(np.mean(np.exp(4j * np.arctan2(v[:, 1], v[:, 0])))) / 4  # modulo 90°
    c, s = np.cos(theta), np.sin(theta)
    xy = xy @ np.array([[c, -s], [s, c]])  # rotate by -theta
    ij = np.round((xy - xy.min(axis=0)) / step).astype(int)
    shape = tuple(ij.max(axis=0) + 1)
    if shape[0] * shape[1] > MAX_RASTER_PIXELS:
        raise ValueError(f"raster {shape} too large (step {step:g}, angle {np.degrees(theta):.1f}°)")
    return ij[:, 0], ij[:, 1], shape


def largest_component(mask):
    lab, n = ndimage.label(mask, structure=np.ones((3, 3)))
    if n == 0:
        return np.zeros_like(mask)
    sizes = ndimage.sum(mask, lab, index=np.arange(1, n + 1))
    return lab == (np.argmax(sizes) + 1)


def geodesic_length(mask):
    """Longest shortest path inside a binary mask (double BFS), in pixels."""
    pts = np.argwhere(mask)
    if len(pts) < 2:
        return 0.0
    index = -np.ones(mask.shape, dtype=np.int64)
    index[tuple(pts.T)] = np.arange(len(pts))
    rows, cols, w = [], [], []
    for dx, dy in [(1, 0), (0, 1), (1, 1), (1, -1)]:
        q = pts + [dx, dy]
        ok = (q[:, 0] >= 0) & (q[:, 0] < mask.shape[0]) & (q[:, 1] >= 0) & (q[:, 1] < mask.shape[1])
        j = np.full(len(pts), -1)
        j[ok] = index[q[ok, 0], q[ok, 1]]
        good = j >= 0
        rows.append(np.flatnonzero(good)); cols.append(j[good])
        w.append(np.full(good.sum(), np.hypot(dx, dy)))
    rows, cols, w = map(np.concatenate, (rows, cols, w))
    G = sparse.coo_matrix((w, (rows, cols)), shape=(len(pts), len(pts))).tocsr()
    d0 = csgraph.dijkstra(G, directed=False, indices=0)
    far = int(np.nanargmax(np.where(np.isfinite(d0), d0, -1)))
    d1 = csgraph.dijkstra(G, directed=False, indices=far)
    return float(np.max(d1[np.isfinite(d1)]))


def section_features(df):
    lab = df.label.str.lower()
    cls = lab.map(DEPTH_CLASS).fillna("ignore")
    counts = cls.value_counts()
    n_cls = sum(counts.get(c, 0) for c in DEPTH_VALUE)
    D = (sum(counts.get(c, 0) * v for c, v in DEPTH_VALUE.items()) / n_cls) if n_cls else np.nan

    ix, iy, shape = rasterize(df)
    tissue = np.zeros(shape, bool)
    t = ~lab.isin(NON_TISSUE).values
    tissue[ix[t], iy[t]] = True
    tissue = largest_component(ndimage.binary_fill_holes(ndimage.binary_closing(tissue, iterations=2)))
    axial = np.zeros(shape, bool)
    a = lab.isin(AXIAL_LABELS).values
    axial[ix[a], iy[a]] = True
    axial = largest_component(ndimage.binary_closing(axial, iterations=2))
    L_body, L_axial = geodesic_length(tissue), geodesic_length(axial)
    # In curled (C-shaped) embryos the longest path through the tissue can short-cut
    # between head and tail, so a full-length spinal cord can exceed it: cap at 1.
    C_raw = L_axial / L_body if L_body > 0 else np.nan
    C = min(1.0, C_raw)

    n = len(df)
    return {
        "n_spots": n,
        "frac_axial": counts.get("axial", 0) / n,
        "frac_paramedian": counts.get("paramedian", 0) / n,
        "frac_lateral": counts.get("lateral", 0) / n,
        "frac_classified": n_cls / n,
        "D_composition": D,
        "C_axial_coverage": C,
        "C_raw": C_raw,
        "body_length_px": L_body,
        "axial_length_px": L_axial,
        "L_laterality": W_COMPOSITION * D + (1 - W_COMPOSITION) * (1 - C),
    }


def side_hint(df, expr):
    """Fraction of heart/gut spots expressing the left-side gene (exploratory)."""
    if expr is None:
        return np.nan, 0
    m = df.label.str.lower().isin(SIDE_ORGANS).values
    return (float((expr[m] > 0).mean()) if m.sum() else np.nan), int(m.sum())

## Run on all available sections

In [ ]:
sections, rows, unknown = {}, [], set()
for ds, (folder, pattern, label_col, regex) in DATASETS.items():
    for path in sorted(folder.glob(pattern)):
        meta = parse_name(path, regex)
        df, genes = read_section(path, label_col)
        unknown |= {(ds, l) for l in df.label.str.lower().unique() if l not in DEPTH_CLASS}
        feats = section_features(df)
        if COMPUTE_SIDE_HINT:
            frac, n_side = side_hint(df, gene_vector(path, SIDE_GENE[ds], genes))
            feats.update({"side_gene_frac": frac, "side_n_spots": n_side})
        key = f"{ds}:{path.stem}"
        sections[key] = df
        rows.append({"key": key, "dataset": ds, "file": path.name, **meta, **feats})
        print(f"done {key}")

scores = pd.DataFrame(rows).set_index("key")
if unknown:
    print("\nLabels not in DEPTH_CLASS (treated as 'ignore'):")
    for ds, l in sorted(unknown):
        print(f"  {ds}: {l!r}")

cols = ["stage", "embryo", "section", "frac_axial", "frac_paramedian", "frac_lateral",
        "frac_classified", "D_composition", "C_axial_coverage", "L_laterality"]
if COMPUTE_SIDE_HINT:
    cols += ["side_gene_frac", "side_n_spots"]
scores[cols].round(3)

done HESTA:CS12-13_E2S1_HESTA
done HESTA:CS12-13_E2S3_HESTA
done HESTA:CS12-13_E2S4_HESTA
done HESTA:CS14-15_E2S10_HESTA
done HESTA:CS14-15_E2S6_HESTA
done HESTA:CS14-15_E2S7_HESTA
done HESTA:CS14-15_E2S8_HESTA
done HESTA:CS14-15_E2S9_HESTA
done HESTA:CS17_E1S2_HESTA
done HESTA:CS17_E1S7_HESTA
done HESTA:CS17_E1S8_HESTA
done HESTA:CS17_E1S9_HESTA
done HESTA:CS18_E1S1_HESTA
done HESTA:CS18_E1S2_HESTA
done HESTA:CS18_E1S3_HESTA
done HESTA:CS18_E1S4_HESTA
done HESTA:CS19_E1S2_HESTA
done HESTA:CS19_E2S2_HESTA
done HESTA:CS19_E2S3_HESTA
done HESTA:CS19_E2S4_HESTA
done HESTA:CS20_E1S5_HESTA
done HESTA:CS20_E1S6_HESTA
done HESTA:CS20_E1S7_HESTA
done HESTA:CS23_E2S5_HESTA
done HESTA:CS23_E2S6_HESTA
done HESTA:CS23_E2S7_HESTA
done HESTA:CS23_E2S8_HESTA
done MOSTA:E10.5_E1S1.MOSTA
done MOSTA:E10.5_E1S2.MOSTA
done MOSTA:E10.5_E2S1.MOSTA


## Ordering within each embryo

Sections of the same embryo, sorted from most medial (low `L`) to most lateral.
This is the most reliable use of the score.

In [ ]:
order = (scores.sort_values(["dataset", "stage", "embryo", "L_laterality"])
               [["dataset", "stage", "embryo", "section", "D_composition",
                 "C_axial_coverage", "L_laterality"]])
order["rank_in_embryo"] = order.groupby(["dataset", "stage", "embryo"]).cumcount() + 1
order.round(3)

## Visual check

Spots coloured by depth class (red = axial, orange = paramedian, blue = lateral,
grey = ignored). The score should match the impression: a continuous red band
(spinal cord) for medial sections, blue limb buds for lateral ones.

In [ ]:
COLORS = {"axial": "#d62728", "paramedian": "#ff7f0e", "lateral": "#1f77b4", "ignore": "#d9d9d9"}
keys = list(scores.sort_values(["dataset", "stage", "L_laterality"]).index)
ncol = 3
nrow = int(np.ceil(len(keys) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 5 * nrow), squeeze=False)
for ax, key in zip(axes.flat, keys):
    df = sections[key]
    cls = df.label.str.lower().map(DEPTH_CLASS).fillna("ignore")
    keep = ~df.label.str.lower().isin(NON_TISSUE)
    for c in ["ignore", "paramedian", "lateral", "axial"]:
        m = keep & (cls == c)
        ax.scatter(df.x[m], df.y[m], s=0.3, c=COLORS[c], rasterized=True)
    r = scores.loc[key]
    ax.set_title(f"{key}\nD={r.D_composition:.2f}  C={r.C_axial_coverage:.2f}  L={r.L_laterality:.2f}",
                 fontsize=9)
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
for ax in axes.flat[len(keys):]:
    ax.axis("off")
handles = [plt.Line2D([], [], marker="o", ls="", color=v, label=k) for k, v in COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(OUT_DIR / "laterality_sections.png", dpi=150)
plt.show()

## Pairing HESTA and MOSTA sections

For every HESTA section, list the MOSTA sections of the matched stage(s) ranked
by |ΔL|. Stages without downloaded MOSTA files are reported as missing.

In [ ]:
def norm_stage(s):
    return s.replace("–", "-")

h = scores[scores.dataset == "HESTA"]
m = scores[scores.dataset == "MOSTA"]
pairs = []
for key, r in h.iterrows():
    targets = STAGE_PAIRS.get(norm_stage(r.stage), [])
    cand = m[m.stage.isin(targets)]
    if cand.empty:
        pairs.append({"HESTA": key, "MOSTA": None, "matched_stages": ",".join(targets) or "?",
                      "L_HESTA": r.L_laterality, "L_MOSTA": np.nan, "abs_dL": np.nan,
                      "note": "no MOSTA file for matched stage(s)"})
        continue
    for mk, mr in cand.iterrows():
        pairs.append({"HESTA": key, "MOSTA": mk, "matched_stages": ",".join(targets),
                      "L_HESTA": r.L_laterality, "L_MOSTA": mr.L_laterality,
                      "abs_dL": abs(r.L_laterality - mr.L_laterality), "note": ""})
pairs = pd.DataFrame(pairs).sort_values(["HESTA", "abs_dL"])
pairs.round(3)

## Save

In [ ]:
scores.to_csv(OUT_DIR / "laterality_scores.csv")
pairs.to_csv(OUT_DIR / "laterality_pairs.csv", index=False)
print("saved to", OUT_DIR.resolve())